# Exercise 04 — SOLUTION: HH + A-Current

Try the stub first! Only open this after attempting [ex04_stub.ipynb](ex04_stub.ipynb).

---

In [ ]:
!nvidia-smi

In [ ]:
%%writefile hh_a_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e));exit(1);}} while(0)

__constant__ float c_Cm,c_gNa,c_gK,c_gL,c_gA,c_ENa,c_EK,c_EL,c_dt;

__device__ __forceinline__ float alpha_m(float V){float d=V+40.f;return fabsf(d)<1e-5f?1.f:0.1f*d/(1.f-expf(-d/10.f));}
__device__ __forceinline__ float beta_m(float V) {return 4.f*expf(-(V+65.f)/18.f);}
__device__ __forceinline__ float alpha_h(float V){return 0.07f*expf(-(V+65.f)/20.f);}
__device__ __forceinline__ float beta_h(float V) {return 1.f/(1.f+expf(-(V+35.f)/10.f));}
__device__ __forceinline__ float alpha_n(float V){float d=V+55.f;return fabsf(d)<1e-5f?0.1f:0.01f*d/(1.f-expf(-d/10.f));}
__device__ __forceinline__ float beta_n(float V) {return 0.125f*expf(-(V+65.f)/80.f);}

// SOLUTION: A-current rate functions
__device__ __forceinline__ float alpha_a(float V){
    float d=V+65.f; return fabsf(d)<1e-5f?0.02f:0.02f*d/(1.f-expf(-d/10.f));}
__device__ __forceinline__ float beta_a(float V) {
    return 0.01f*(V+65.f)*expf(-(V+65.f)/10.f);}
__device__ __forceinline__ float alpha_b(float V){return 0.003f*expf(-(V+65.f)/30.f);}
__device__ __forceinline__ float beta_b(float V) {return 0.015f/(1.f+expf(-(V+35.f)/10.f));}

__device__ __forceinline__ void hh_deriv_A(
    float V,float m,float h,float n,float a,float b,float I,
    float* dV,float* dm,float* dh,float* dn,float* da,float* db
){
    float INa=c_gNa*m*m*m*h*(V-c_ENa);
    float IK =c_gK*n*n*n*n*(V-c_EK);
    float IL =c_gL*(V-c_EL);
    float IA =c_gA*a*a*a*b*(V-c_EK);  // A-current uses EK reversal
    *dV=(I-INa-IK-IL-IA)/c_Cm;
    *dm=alpha_m(V)*(1.f-m)-beta_m(V)*m;
    *dh=alpha_h(V)*(1.f-h)-beta_h(V)*h;
    *dn=alpha_n(V)*(1.f-n)-beta_n(V)*n;
    *da=alpha_a(V)*(1.f-a)-beta_a(V)*a;
    *db=alpha_b(V)*(1.f-b)-beta_b(V)*b;
}

__global__ void hh_a_sweep(const float* IA,int* sc,int N,int T,int skip){
    int i=blockIdx.x*blockDim.x+threadIdx.x;
    if(i>=N)return;
    float V=-65.f,I=IA[i];
    float am=alpha_m(V),bm=beta_m(V),ah=alpha_h(V),bh=beta_h(V),an=alpha_n(V),bn=beta_n(V);
    float aa=alpha_a(V),ba=beta_a(V),ab=alpha_b(V),bb=beta_b(V);
    float m=am/(am+bm),h=ah/(ah+bh),n=an/(an+bn);
    float a=aa/(aa+ba),b=ab/(ab+bb);
    int spk=0,abv=0;
    for(int s=0;s<T;s++){
        float dV,dm,dh,dn,da,db;
        hh_deriv_A(V,m,h,n,a,b,I,&dV,&dm,&dh,&dn,&da,&db);
        V+=c_dt*dV;
        m=fmaxf(0.f,fminf(1.f,m+c_dt*dm)); h=fmaxf(0.f,fminf(1.f,h+c_dt*dh));
        n=fmaxf(0.f,fminf(1.f,n+c_dt*dn)); a=fmaxf(0.f,fminf(1.f,a+c_dt*da));
        b=fmaxf(0.f,fminf(1.f,b+c_dt*db));
        if(s>=skip){if(V>0.f&&!abv){spk++;abv=1;} if(V<-30.f)abv=0;}
    }
    sc[i]=spk;
}

int main(){
    const int N=2000; float T_ms=500.f,T_skip=100.f,dt=0.01f,gA=4.f;
    float Cm=1.f,gNa=120.f,gK=36.f,gL=0.3f,ENa=50.f,EK=-77.f,EL=-54.4f;
    CUDA_CHECK(cudaMemcpyToSymbol(c_Cm,&Cm,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_gNa,&gNa,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_gK,&gK,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_gL,&gL,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_gA,&gA,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_ENa,&ENa,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_EK,&EK,4)); CUDA_CHECK(cudaMemcpyToSymbol(c_EL,&EL,4));
    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,&dt,4));
    float *hI=(float*)malloc(N*4);
    for(int i=0;i<N;i++) hI[i]=30.f*i/(N-1.f);
    float *dI; int *dsc;
    CUDA_CHECK(cudaMalloc(&dI,N*4)); CUDA_CHECK(cudaMalloc(&dsc,N*4));
    CUDA_CHECK(cudaMemcpy(dI,hI,N*4,cudaMemcpyHostToDevice));
    int thr=256,blk=(N+thr-1)/thr,Ts=(int)(T_ms/dt),sk=(int)(T_skip/dt);
    hh_a_sweep<<<blk,thr>>>(dI,dsc,N,Ts,sk);
    CUDA_CHECK(cudaDeviceSynchronize());
    int *hsc=(int*)malloc(N*4);
    CUDA_CHECK(cudaMemcpy(hsc,dsc,N*4,cudaMemcpyDeviceToHost));
    FILE* f=fopen("fi_with_A.txt","w");
    float Teff=(T_ms-T_skip)/1e3f;
    for(int i=0;i<N;i++) fprintf(f,"%.4f %.2f\n",hI[i],hsc[i]/Teff);
    fclose(f);
    printf("Done. fi_with_A.txt written.\n");
    free(hI);free(hsc);cudaFree(dI);cudaFree(dsc);
    return 0;
}

In [ ]:
!nvcc -O2 -o hh_a_sol hh_a_sol.cu -lm && ./hh_a_sol

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d=np.loadtxt('fi_with_A.txt')
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(d[:,0],d[:,1],'r-',lw=2,label='HH + A-current (gA=4)')
ax.set_xlabel('Input current (μA/cm²)',fontsize=13)
ax.set_ylabel('Firing rate (Hz)',fontsize=13)
ax.set_title('HH f-I curve with A-current',fontsize=13)
ax.legend(fontsize=11); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()
rheobase=d[d[:,1]>0,0][0] if (d[:,1]>0).any() else None
print(f"Rheobase with I_A: {rheobase:.2f} μA/cm² (higher than {6.26:.2f} without I_A)")